# lightrag-langchain 完整演示

本 Notebook 演示 **lightrag-langchain** 全部六种查询模式：Naive、Local、Global、Hybrid、Mix、Bypass。

lightrag-langchain 是一个基于 LangChain 框架的只读查询库，直接读取 LightRAG 已处理好的 PostgreSQL 知识图谱数据库。它不依赖 LightRAG 运行时，提供标准的 LangChain Retriever + Chain 接口。

## 前置条件

- PostgreSQL 数据库已由上游 LightRAG 实例完成知识图谱构建（含 `entities_vdb`、`relationships_vdb`、`chunks_vdb` 和 AGE 图数据）
- `.env` 文件已配置完成（参考 `.env.example`）
- 依赖已安装（`uv sync`）

In [ ]:
# Cell 1: Imports and settings check
import sys
from pathlib import Path

# If running from examples/ directory, add project root
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'examples' else Path.cwd()))

from lightrag_langchain import (
    NaiveChain, NaiveRetriever,
    LocalChain, LocalRetriever,
    GlobalChain, GlobalRetriever,
    HybridChain, HybridRetriever,
    MixChain, MixRetriever,
    BypassChain, BypassRetriever,
    create_llm, create_embedding,
)
from lightrag_langchain.config import settings
from lightrag_langchain.data.graph import PGGraphStore
from lightrag_langchain.data.store import PGVectorStore

print(f"PG   : {settings.pg.host}:{settings.pg.port}/{settings.pg.database}")
print(f"LLM  : {settings.llm.model}")
print(f"Embed: {settings.embedding.model} (dim={settings.embedding.dim})")
print(f"Top-K: {settings.query_params.top_k}, Chunk-Top-K: {settings.query_params.chunk_top_k}")
print("✓ Settings loaded successfully")

In [ ]:
# Cell 2: Create shared data connections, LLM, and embedding

# Vector store (for vector similarity search on entities_vdb, relationships_vdb, chunks_vdb)
vector_store = PGVectorStore(
    embedding_dim=settings.embedding.dim,
    host=settings.pg.host,
    port=settings.pg.port,
    user=settings.pg.user,
    password=settings.pg.password.get_secret_value(),
    database=settings.pg.database,
)

# Graph store (for Apache AGE entity/relationship graph lookups)
graph_store = PGGraphStore(
    host=settings.pg.host,
    port=settings.pg.port,
    user=settings.pg.user,
    password=settings.pg.password.get_secret_value(),
    database=settings.pg.database,
    workspace=settings.pg.workspace,
)

# LLM and embedding factories (lazy proxies)
llm = create_llm(settings.llm)
embedding = create_embedding(settings.embedding)

print("✓ PGVectorStore created")
print("✓ PGGraphStore created")
print("✓ LLM created (model={})".format(settings.llm.model))
print("✓ Embedding created (model={})".format(settings.embedding.model))

## 1. Naive 模式 — 纯向量搜索

Naive 模式对 `chunks_vdb` 表执行纯向量相似度搜索，不涉及图遍历。这是最简单的查询模式，适用于语义匹配型查询。

In [ ]:
# Naive mode: pure vector chunk search
naive_retriever = NaiveRetriever(
    vector_store=vector_store,
    embedding_config=settings.embedding,
)
naive_chain = NaiveChain(retriever=naive_retriever, llm=llm)

question = "启动东莞市防风Ⅰ级应急响应后需要落实哪些措施？"
result = await naive_chain.ainvoke(question)

print(f"✓ 模式: {result['mode']}")
print(f"✓ 关键词: {result['keywords']}")
print(f"✓ 来源数: {len(result['sources'])}")
print(f"---回答---\n{result['answer']}")

## 2. Local 模式 — 实体中心图扩展

Local 模式先对 `entities_vdb` 执行向量搜索获取 Top-K 实体，再通过 Apache AGE 图数据库扩展获取关联的边和邻居实体。适用于实体级的深度查询。

In [ ]:
# Local mode: entity-centric graph traversal
local_retriever = LocalRetriever(
    vector_store=vector_store,
    graph_store=graph_store,
    embedding_config=settings.embedding,
)
local_chain = LocalChain(retriever=local_retriever, llm=llm)

question = "珠江流域超标准洪水时水库抢险标准是什么？"
result = await local_chain.ainvoke(question)

print(f"✓ 模式: {result['mode']}")
print(f"✓ 关键词: {result['keywords']}")
print(f"✓ 来源数: {len(result['sources'])}")
print(f"---回答---\n{result['answer']}")

## 3. Global 模式 — 关系中心图扩展

Global 模式先对 `relationships_vdb` 执行向量搜索获取 Top-K 关系，再通过 Apache AGE 图数据库查找关联的实体。适用于关系级别的宏观查询。

In [ ]:
# Global mode: relation-centric graph traversal
global_retriever = GlobalRetriever(
    vector_store=vector_store,
    graph_store=graph_store,
    embedding_config=settings.embedding,
)
global_chain = GlobalChain(retriever=global_retriever, llm=llm)

question = "哪些机构参与了防汛应急响应？"
result = await global_chain.ainvoke(question)

print(f"✓ 模式: {result['mode']}")
print(f"✓ 关键词: {result['keywords']}")
print(f"✓ 来源数: {len(result['sources'])}")
print(f"---回答---\n{result['answer']}")

## 4. Hybrid 模式 — 并行 local + global

Hybrid 模式并行执行 Local 和 Global 检索，结果按 round-robin 方式交错合并。适用于需要宏观和微观兼顾的综合查询。

In [ ]:
# Hybrid mode: parallel local + global retrieval with round-robin merge
hybrid_retriever = HybridRetriever(
    vector_store=vector_store,
    graph_store=graph_store,
    embedding_config=settings.embedding,
)
hybrid_chain = HybridChain(retriever=hybrid_retriever, llm=llm)

question = "防风应急响应和防汛应急响应有何异同？"
result = await hybrid_chain.ainvoke(question)

print(f"✓ 模式: {result['mode']}")
print(f"✓ 关键词: {result['keywords']}")
print(f"✓ 来源数: {len(result['sources'])}")
print(f"---回答---\n{result['answer']}")

## 5. Mix 模式 — hybrid + chunk 搜索

Mix 模式在 Hybrid 检索的基础上追加 `chunks_vdb` 向量搜索，融合图知识和原始文本块信息。适用于力求最大覆盖的全量检索。

In [ ]:
# Mix mode: hybrid retrieval + chunk vector search
mix_retriever = MixRetriever(
    vector_store=vector_store,
    graph_store=graph_store,
    embedding_config=settings.embedding,
)
mix_chain = MixChain(retriever=mix_retriever, llm=llm)

question = "洪水防汛应急响应的完整体系是什么？"
result = await mix_chain.ainvoke(question)

print(f"✓ 模式: {result['mode']}")
print(f"✓ 关键词: {result['keywords']}")
print(f"✓ 来源数: {len(result['sources'])}")
print(f"---回答---\n{result['answer']}")

## 6. Bypass 模式 — 直接 LLM 调用

Bypass 模式跳过所有检索步骤（无关键词提取、无向量搜索、无 token 预算控制），直接将用户问题发送给 LLM。适用于无需外部知识的纯对话场景。

In [ ]:
# Bypass mode: direct LLM call, no retrieval
bypass_retriever = BypassRetriever()
bypass_chain = BypassChain(retriever=bypass_retriever, llm=llm)

question = "请简要介绍中国的应急管理体系。"
result = await bypass_chain.ainvoke(question)

print(f"✓ 模式: {result['mode']}")
print(f"✓ 关键词: {result['keywords']}")
print(f"✓ 来源数: {len(result['sources'])}")
print(f"---回答---\n{result['answer']}")

## 总结

| 模式 | 检索策略 | 需要图存储 | 适用场景 |
|------|----------|------------|----------|
| Naive | 纯向量搜索 chunks_vdb | 否 | 简单语义匹配 |
| Local | entities_vdb + AGE 图扩展 | 是 | 实体级深度查询 |
| Global | relationships_vdb + AGE 图查找 | 是 | 关系级宏观查询 |
| Hybrid | 并行 local + global，round-robin 合并 | 是 | 宏微观兼顾 |
| Mix | hybrid + chunks_vdb 向量搜索 | 是 | 最大覆盖全量检索 |
| Bypass | 直接 LLM 调用 | 否 | 无需外部知识对话 |

lightrag-langchain 提供了完整的 LangChain 接口，可以无缝集成到 LangGraph、LangServe 等 LangChain 生态工具中。每种查询模式均提供独立的 Retriever 和 Chain​​​ ，可单独使用或组合编排。